# Section 1: Extract US Politics Sequences from WikiText-103

In [1]:
import re
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

NUM_SEQUENCES   = 50
NUM_PARAPHRASES = 10   # M=10; original is paraphrase 0, so 9 new paraphrases generated
SEQ_LEN         = 100  # exact token length for every row
SEED            = 42
SAVE_PATH       = "wikitext_paraphrase"
CONTROL_SAVE_PATH = "wikitext_paraphrase_control"

In [2]:
_US_POLITICS_ANCHORS = [
    "congress", "senate", "republican", "democrat", "white house",
    "house of representatives", "obama", "trump", "biden", "clinton",
    "gop", "democratic party", "u.s. president", "u.s. congress",
    "u.s. senate", "midterm", "filibuster", "oval office",
]

_US_POLITICS_KEYWORDS = [
    "election", "president", "legislation", "political party",
    "congressman", "senator", "governor", "ballot", "partisan",
    "bipartisan", "caucus", "campaign", "polling", "primaries",
    "inauguration", "executive order", "speaker of the house",
]


def _is_us_politics(text: str) -> bool:
    """Return True if text has >= 2 anchor keyword hits AND >= 5 total keyword hits."""
    lower = text.lower()
    anchor_hits = sum(1 for kw in _US_POLITICS_ANCHORS if kw in lower)
    if anchor_hits < 3:
        return False
    support_hits = sum(1 for kw in _US_POLITICS_KEYWORDS if kw in lower)
    return (anchor_hits + support_hits) >= 6

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
dataset   = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
dataset   = dataset.filter(lambda x: not re.match(r"(?: =)+[^=]+(?:= )+", x["text"]), desc="Filtering headers", num_proc=16)
dataset   = dataset.filter(lambda x: _is_us_politics(x["text"]), desc="Filtering by topic", num_proc=16)
dataset   = dataset.filter(lambda x: len(tokenizer(x["text"])["input_ids"]) > (SEQ_LEN + 50), desc="Filtering by min length", num_proc=16)
dataset   = dataset.shuffle(seed=SEED)

# Take first NUM_SEQUENCES for paraphrase dataset; the rest available for control
sequences = []
all_filtered = list(dataset)
for example in tqdm(all_filtered[:NUM_SEQUENCES], desc="Collecting paraphrase source sequences"):
    tokens = tokenizer.encode(example["text"])
    sequences.append({
        "seq_idx":   len(sequences),
        "tokens":    tokens,            # full tokens — truncated to SEQ_LEN only at save time
        "full_text": example["text"],   # full original text for paraphrase generation
    })

print(f"Collected {len(sequences)} paraphrase source sequences")
print(f"Remaining for control: {len(all_filtered) - NUM_SEQUENCES} sequences")

Filtering headers (num_proc=16):   0%|          | 0/1801350 [00:00<?, ? examples/s]

Filtering by topic (num_proc=16):   0%|          | 0/1496276 [00:00<?, ? examples/s]

Filtering by min length (num_proc=16):   0%|          | 0/962 [00:00<?, ? examples/s]

Collected 50 paraphrase source sequences
Remaining for control: 606 sequences


# Section 2: Paraphrase Sequences with OpenAI API

In [4]:
import hashlib, json, os, time
from openai import OpenAI
from tqdm.notebook import tqdm
from dotenv import load_dotenv

OPENAI_MODEL = "gpt-5-mini-2025-08-07"
CACHE_PATH   = "paraphrase_cache.json"

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [5]:
def paraphrase(text, n=NUM_PARAPHRASES - 1):
    """Generate n paraphrases of text (original occupies slot 0)."""
    prompt = (
        f"Paraphrase the following text {n} times. "
        f"Write paraphrases of similar length (at least 100 words). "
        f"Return ONLY a JSON object with key 'paraphrases' whose value is an array of {n} strings. "
        f"\n\nText: {text}"
    )
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            result =  json.loads(resp.choices[0].message.content)["paraphrases"]
            if len(result) != n:
                raise ValueError(f"Expected {n} paraphrases but got {len(result)}")
            return result
        except Exception as e:
            time.sleep(2 ** attempt)
    raise RuntimeError("Failed after 5 retries")

# Section 3: Assemble Paraphrase Dataset (N×M rows, 100 tokens each)

In [6]:
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
else:
    cache = {}

rows = []
for seq in tqdm(sequences, desc="Paraphrasing"):
    key = hashlib.sha256(f"{seq['full_text']}|{OPENAI_MODEL}".encode()).hexdigest()

    expected = NUM_PARAPHRASES - 1
    if key in cache and len(cache[key]) < expected:
        print(f"Cache entry {key[:8]}… has {len(cache[key])} paraphrases (expected {expected}), regenerating.")
        del cache[key]

    # paraphrase_idx = 0: original, truncated to SEQ_LEN
    orig_ids = seq["tokens"][:SEQ_LEN]
    rows.append({
        "input_ids":      orig_ids,
        "attention_mask": [1] * SEQ_LEN,
        "text":           tokenizer.decode(orig_ids),
        "source_idx":     seq["seq_idx"],
        "paraphrase_idx": 0,
    })

    # paraphrase_idx = 1..M-1: generated paraphrases
    if key not in cache:
        cache[key] = paraphrase(seq["full_text"])
        with open(CACHE_PATH, "w") as f:
            json.dump(cache, f, indent=2)
    paraphrases = cache[key]

    added = 0
    attempt_pool = list(paraphrases)
    para_idx = 1
    pool_pos = 0
    MAX_RETRIES = 3
    retries = 0
    while added < NUM_PARAPHRASES - 1:
        if pool_pos >= len(attempt_pool):
            if retries >= MAX_RETRIES:
                raise RuntimeError(
                    f"seq_idx={seq['seq_idx']}: could not collect enough valid paraphrases "
                    f"after {MAX_RETRIES} retries (got {added}/{NUM_PARAPHRASES - 1})"
                )
            print(f"Pool exhausted with {added}/{NUM_PARAPHRASES - 1} paraphrases, retrying ({retries + 1}/{MAX_RETRIES})…")
            attempt_pool.extend(paraphrase(seq["full_text"]))
            retries += 1
        text = attempt_pool[pool_pos]
        pool_pos += 1
        ids = tokenizer.encode(text)
        if len(ids) < SEQ_LEN:
            continue  # skip too-short result, try next
        rows.append({
            "input_ids":      ids[:SEQ_LEN],
            "attention_mask": [1] * SEQ_LEN,
            "text":           tokenizer.decode(ids[:SEQ_LEN]),
            "source_idx":     seq["seq_idx"],
            "paraphrase_idx": para_idx,
        })
        para_idx += 1
        added += 1

from datasets import Dataset
ds = Dataset.from_list(rows)
ds.save_to_disk(SAVE_PATH)
print(f"Saved {len(ds)} rows to {SAVE_PATH}/")
print(f"Target: {NUM_SEQUENCES * NUM_PARAPHRASES} rows")

Paraphrasing:   0%|          | 0/50 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saved 500 rows to wikitext_paraphrase/
Target: 500 rows


In [7]:
from collections import defaultdict
from datasets import load_from_disk

ds_val = load_from_disk(SAVE_PATH)

# Check all input_ids have length 100
bad = [i for i, row in enumerate(ds_val) if len(row["input_ids"]) != SEQ_LEN]
assert not bad, f"Rows with wrong length: {bad[:5]}"
print(f"[PASS] All {len(ds_val)} rows have input_ids of length {SEQ_LEN}.")

groups = defaultdict(list)
for row in ds_val:
    groups[row["source_idx"]].append(row)

for src_idx, grp in groups.items():
    para_idxs = {r["paraphrase_idx"] for r in grp}
    assert 0 in para_idxs, f"source_idx {src_idx}: missing original (paraphrase_idx=0)"
print(f"[PASS] All {len(groups)} source groups contain the original (paraphrase_idx=0).")
print(f"Total rows: {len(ds_val)} (target: {NUM_SEQUENCES * NUM_PARAPHRASES})")

[PASS] All 500 rows have input_ids of length 100.
[PASS] All 50 source groups contain the original (paraphrase_idx=0).
Total rows: 500 (target: 500)


# Section 4: Control Dataset (N×M unique WikiText sequences, no paraphrases)

In [8]:
# Use sequences after the first NUM_SEQUENCES (disjoint from paraphrase sources)
control_target = NUM_SEQUENCES * NUM_PARAPHRASES  # 500
control_pool = all_filtered[NUM_SEQUENCES:]  # disjoint from paraphrase sources

control_rows = []
for example in tqdm(control_pool, desc="Collecting control sequences"):
    if len(control_rows) >= control_target:
        break
    tokens = tokenizer.encode(example["text"])
    if len(tokens) < SEQ_LEN:
        continue
    control_rows.append({
        "input_ids":      tokens[:SEQ_LEN],
        "attention_mask": [1] * SEQ_LEN,
        "text": example["text"]
    })

print(f"Collected {len(control_rows)} control rows (target: {control_target})")

from datasets import Dataset
ds_control = Dataset.from_list(control_rows[:control_target])
ds_control.save_to_disk(CONTROL_SAVE_PATH)
print(f"Saved {len(ds_control)} rows to {CONTROL_SAVE_PATH}/")

Collected 500 control rows (target: 500)


Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saved 500 rows to wikitext_paraphrase_control/


In [9]:
from datasets import load_from_disk

ds_ctrl_val = load_from_disk(CONTROL_SAVE_PATH)

assert len(ds_ctrl_val) == NUM_SEQUENCES * NUM_PARAPHRASES, \
    f"Expected {NUM_SEQUENCES * NUM_PARAPHRASES} rows, got {len(ds_ctrl_val)}"
print(f"[PASS] Control dataset has {len(ds_ctrl_val)} rows.")

bad = [i for i, row in enumerate(ds_ctrl_val) if len(row["input_ids"]) != SEQ_LEN]
assert not bad, f"Rows with wrong length: {bad[:5]}"
print(f"[PASS] All rows have input_ids of length {SEQ_LEN}.")

[PASS] Control dataset has 500 rows.
[PASS] All rows have input_ids of length 100.
